In [1]:
# ==========================================================
# ACTIVIDAD 14. Muestra definitiva y separación de empresas
# Excluye los 2 errores de datos demostrados, sortea las
# empresas en entrenamiento (70 %) y reservadas (30 %) dentro
# de cada estrato y guarda los precios de la muestra.
# ==========================================================
from pathlib import Path

import pandas as pd
from google.colab import drive
from sklearn.model_selection import train_test_split

SEMILLA = 42                      # fija el sorteo: siempre salen las mismas empresas
PCT_RESERVADAS = 0.30
FIN_ENTRENAMIENTO = pd.Timestamp("2022-12-30")
FIN_VALIDACION = pd.Timestamp("2024-12-31")
ERRORES_DEMOSTRADOS = {
    "CHRD": "Empalme de acciones canceladas y nuevas tras el Capítulo 11 (20-nov-2020)",
    "CENN": "Contrasplit aplicado en fecha equivocada (1-dic-2023 en lugar de 8-dic-2023)",
}

drive.mount("/content/drive")
CARPETA = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo/00_control")
CARPETA_DATOS = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo/01_datos")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# ----------------------------------------------------------
# 1. Elegibles menos los errores demostrados
# ----------------------------------------------------------
elegibles = pd.read_csv(CARPETA / "elegibles_muestra.csv", dtype={"ticker": str, "cik": str},
                        keep_default_na=False, na_values=[""]).set_index("ticker")
print("Elegibles:", len(elegibles))
faltan = set(ERRORES_DEMOSTRADOS) - set(elegibles.index)
assert not faltan, f"No están en las elegibles: {faltan}"
muestra = elegibles.drop(index=list(ERRORES_DEMOSTRADOS)).copy()
print("Excluidas por error de datos:", list(ERRORES_DEMOSTRADOS), "-> muestra:", len(muestra))

# ----------------------------------------------------------
# 2. Sorteo estratificado de empresas
# ----------------------------------------------------------
# Estrato = liquidez x volatilidad (9 combinaciones, calculadas solo con entrenamiento)
muestra["estrato"] = muestra["liquidez"] + " liq | " + muestra["volatilidad"] + " vol"
entrenamiento, reservadas = train_test_split(
    muestra, test_size=PCT_RESERVADAS, stratify=muestra["estrato"], random_state=SEMILLA)
muestra["grupo"] = "entrenamiento"
muestra.loc[reservadas.index, "grupo"] = "reservada"

print("\n=== EMPRESAS POR GRUPO ===")
print(muestra["grupo"].value_counts())
print("\nCIK en los dos grupos a la vez:",
      len(set(muestra.loc[muestra["grupo"] == "entrenamiento", "cik"])
          & set(muestra.loc[muestra["grupo"] == "reservada", "cik"])))

print("\n=== EMPRESAS POR ESTRATO Y GRUPO ===")
tabla = pd.crosstab(muestra["estrato"], muestra["grupo"], margins=True, margins_name="Total")
tabla["% reservadas"] = tabla["reservada"] / tabla["Total"]
print(tabla)

# ----------------------------------------------------------
# 3. ¿Los dos grupos tienen el mismo perfil?
# ----------------------------------------------------------
print("\n=== PERFIL DE CADA GRUPO (medianas en el entrenamiento) ===")
perfil = muestra.groupby("grupo").agg(
    empresas=("cik", "size"),
    volumen_M=("volumen_dolares_mediano", "median"),
    volatilidad_anual=("volatilidad_anual", "median"),
    con_salto_extremo=("saltos_extremos", lambda s: (s > 0).mean()),
)
perfil["volumen_M"] /= 1e6
print(perfil)

print("\n=== DIVISIÓN SIC POR GRUPO (% de empresas) ===")
print((pd.crosstab(muestra["division_sic"], muestra["grupo"], normalize="columns") * 100).round(1))

# ----------------------------------------------------------
# 4. Precios de la muestra con su grupo y su bloque temporal
# ----------------------------------------------------------
precios = pd.read_parquet(CARPETA_DATOS / "precios_universo.parquet")
precios = precios[precios["ticker"].isin(muestra.index)].copy()
precios["ticker"] = precios["ticker"].astype(str)
precios["grupo"] = precios["ticker"].map(muestra["grupo"])
precios["bloque"] = "prueba"
precios.loc[precios["date"] <= FIN_VALIDACION, "bloque"] = "validacion"
precios.loc[precios["date"] <= FIN_ENTRENAMIENTO, "bloque"] = "entrenamiento"
precios = precios.sort_values(["ticker", "date"]).reset_index(drop=True)
for col in ["ticker", "grupo", "bloque"]:
    precios[col] = precios[col].astype("category")

print("\n=== PRECIOS DE LA MUESTRA ===")
print("Filas:", len(precios), "| esperadas:", len(muestra) * 1714)
print(pd.crosstab(precios["grupo"], precios["bloque"], margins=True, margins_name="Total"))

# ----------------------------------------------------------
# 5. Guardar
# ----------------------------------------------------------
columnas = ["cik", "grupo", "estrato", "liquidez", "volatilidad", "division_sic",
            "volumen_dolares_mediano", "volatilidad_anual", "saltos_extremos"]
muestra[columnas].to_csv(CARPETA / "muestra_empresas.csv", index_label="ticker")
pd.DataFrame({"ticker": list(ERRORES_DEMOSTRADOS), "motivo": list(ERRORES_DEMOSTRADOS.values())}
             ).to_csv(CARPETA / "excluidas_error_datos.csv", index=False)
precios.to_parquet(CARPETA_DATOS / "precios_muestra.parquet", index=False)
print("\nGuardado: muestra_empresas.csv, excluidas_error_datos.csv y precios_muestra.parquet")

Mounted at /content/drive
Elegibles: 1593
Excluidas por error de datos: ['CHRD', 'CENN'] -> muestra: 1591

=== EMPRESAS POR GRUPO ===
grupo
entrenamiento    1113
reservada         478
Name: count, dtype: int64

CIK en los dos grupos a la vez: 0

=== EMPRESAS POR ESTRATO Y GRUPO ===
grupo                  entrenamiento  reservada  Total  % reservadas
estrato                                                             
alta liq | alta vol               73         31    104         0.298
alta liq | baja vol              197         85    282         0.301
alta liq | media vol             102         43    145         0.297
baja liq | alta vol              174         75    249         0.301
baja liq | baja vol               70         30    100         0.300
baja liq | media vol             127         54    181         0.298
media liq | alta vol             123         53    176         0.301
media liq | baja vol             104         45    149         0.302
media liq | media vol      